# Union-Find (Disjoint Set Union)

Tracks a collection of non-overlapping sets. Supports two operations:

| Operation | Description | Time (optimized) |
|-----------|-------------|------------------|
| find(x) | Which set does x belong to? | O(α(n)) ≈ O(1) |
| union(x, y) | Merge the sets containing x and y | O(α(n)) ≈ O(1) |

α(n) = inverse Ackermann function - grows so slowly it's effectively constant. For any practical input, α(n) <= 4. See [Analysis of Algorithms](../analysis/01-notation.ipynb) for a detailed explanation.

**Applications:** Kruskal's MST, cycle detection in undirected graphs, connected components, network connectivity.

## Key Optimizations

1. **Union by rank** - attach the shorter tree under the taller one → keeps trees flat
2. **Path compression** - during `find`, point every node directly to the root → flattens on access

Without optimizations: O(n) per operation. With both: O(α(n)) amortized.

> **Procedural style:** The data structure is just two arrays (`parent` and `rank`). Functions operate on them directly - no wrapper class needed.

# Naive Union-Find

Each set is a tree of parent pointers, and the root doubles as the set's identity. `find`
climbs to the root; `union` finds both roots and points one at the other.

Nothing constrains *which* root ends up under which, so a run of unions can build a
single chain - 0 → 1 → 2 → 3 - and `find` degrades to O(n), no better than scanning a
list.

### Why union by rank keeps height O(log n)

A tree of rank k has at least 2^k nodes (provable by induction: rank only increases when
merging two trees of equal rank, doubling the minimum node count). Since 2^k ≤ n, we get
k ≤ log₂n. Every find follows a path from a node to the root, so find is O(log n).

**Time:** O(n) worst case per operation, O(log n) once union by rank is added &nbsp;
**Space:** O(n)

In [ ]:
def make_set_naive(n):
    """Each element is its own parent."""
    return list(range(n))

def find_naive(parent, x):
    """Follow parent pointers to root. O(n) worst case."""
    while parent[x] != x:
        x = parent[x]
    return x

def union_naive(parent, x, y):
    """Make root of x point to root of y."""
    rx, ry = find_naive(parent, x), find_naive(parent, y)
    if rx != ry:
        parent[rx] = ry

def test_naive():
    p = make_set_naive(5)  # {0}, {1}, {2}, {3}, {4}
    union_naive(p, 0, 1)
    union_naive(p, 2, 3)
    assert find_naive(p, 0) == find_naive(p, 1)
    assert find_naive(p, 0) != find_naive(p, 2)
    union_naive(p, 1, 3)  # merge {0,1} and {2,3}
    assert find_naive(p, 0) == find_naive(p, 3)

test_naive()

# Optimized: Union by Rank + Path Compression

Two independent fixes, and the pair is what makes union-find fast.

**Union by rank** attaches the shorter tree under the taller one, where `rank` is an upper
bound on height. A tree can only get taller when two equal-rank trees merge, and that
requires the set to double in size - so height stays O(log n).

**Path compression** flattens as it reads: on the way back out of the recursion, `find`
repoints every node it walked straight at the root, so the next `find` on any of them is
a single hop.

![Path Compression](images/union-find-path-compression.png)

Together they give O(α(n)) amortized per operation - effectively O(1).

Compression means ranks stop being exact heights, which is harmless: they are only used
as a merge heuristic, never as a measurement.

**Time:** O(α(n)) amortized &nbsp; **Space:** O(n)

In [ ]:
def make_set(n):
    parent = list(range(n))
    rank = [0] * n
    return parent, rank

def find(parent, x):
    """Find with path compression. O(α(n)) amortized."""
    if parent[x] != x:
        parent[x] = find(parent, parent[x])  # path compression
    return parent[x]

def union(parent, rank, x, y):
    """Union by rank. O(α(n)) amortized."""
    rx, ry = find(parent, x), find(parent, y)
    if rx == ry:
        return False  # already in same set
    # attach shorter tree under taller
    if rank[rx] < rank[ry]:
        parent[rx] = ry
    elif rank[rx] > rank[ry]:
        parent[ry] = rx
    else:
        parent[ry] = rx
        rank[rx] += 1
    return True

def connected(parent, x, y):
    """Check if x and y are in the same set."""
    return find(parent, x) == find(parent, y)

def test_optimized():
    p, r = make_set(6)
    union(p, r, 0, 1)
    union(p, r, 2, 3)
    union(p, r, 4, 5)
    assert connected(p, 0, 1)
    assert not (connected(p, 0, 2))
    union(p, r, 1, 3)  # merge {0,1} and {2,3}
    assert connected(p, 0, 3)
    assert not (connected(p, 0, 5))

test_optimized()

# Application: Cycle Detection in Undirected Graph

Process edges one at a time. If both endpoints already `find` to the same root, an
earlier chain of edges already connected them, so this edge closes a cycle. Otherwise
merge the two sets and move on.

No recursion over the graph and no adjacency list - which makes this the natural choice
when edges arrive as a stream, and the basis of Kruskal's MST algorithm (take each edge
unless it would form a cycle).

**Time:** O(E · α(V)) &nbsp; **Space:** O(V)

In [ ]:
def has_cycle(n, edges):
    """Detect cycle in undirected graph using Union-Find. Time: O(E × α(V))"""
    parent, rank = make_set(n)
    for u, v in edges:
        if connected(parent, u, v):
            return True
        union(parent, rank, u, v)
    return False

def test_cycle():
    # 0-1-2-0 → cycle
    assert has_cycle(3, [(0, 1), (1, 2), (2, 0)])
    # 0-1, 0-2 → no cycle (tree)
    assert not (has_cycle(3, [(0, 1), (0, 2)]))

test_cycle()

# Application: Count Connected Components

Union every edge, then count the **distinct roots** - one per surviving set. Calling
`find` inside the count also compresses the last paths, so the roots come back cheaply.

The alternative is the BFS/DFS sweep in the [traversal notebook](graph-traversal.ipynb).
Same answer, different trade-off: traversal needs the adjacency list up front, union-find
only needs the edges, one at a time.

**Time:** O(V + E · α(V)) &nbsp; **Space:** O(V)

In [ ]:
def count_components(n, edges):
    """Count connected components. Time: O(E × α(V))"""
    parent, rank = make_set(n)
    for u, v in edges:
        union(parent, rank, u, v)
    # count distinct roots
    return len(set(find(parent, i) for i in range(n)))

def test_components():
    # 0-1-2, 3-4 → 2 components
    assert count_components(5, [(0, 1), (1, 2), (3, 4)]) == 2
    # all connected
    assert count_components(3, [(0, 1), (1, 2)]) == 1
    # no edges
    assert count_components(4, []) == 4

test_components()